# S2UT pilot：実データ診断と3条件の機能検証

固定した少量train/devで、CUDA接続 → T2TT → alignment診断 → strongレシピの3条件 → teacher-free dev評価を実行します。

| 設定 | 既定値 |
|---|---|
| train / dev | 最大10,000 / 200 pair（元splitを保持） |
| T2TT / S2UT | 20,000 / 10,000更新 |
| annealing終了 | 5,000更新（後半はbeta=0） |
| 比較条件 | Baseline / Annealed Monotonic Guidance / Annealed T2TT Guidance |

> **用途**：旧standalone trainerの診断用です。benchmarkの学習済みBaselineとの本比較・ZIP保存と復元には[benchmark比較ノートブック](https://colab.research.google.com/github/Yaaamashiro/alignment-transfer-s2ut/blob/main/notebooks/colab/benchmark_comparison.ipynb)を使います。このpilotはDriveへ直接保存します。

GPUランタイムで上から実行します。コーパスは読み取り専用、testは不使用です。同じ設定・ROOTで再実行すると最後のcheckpointから再開します。

[詳しい手順と判断基準](https://github.com/Yaaamashiro/alignment-transfer-s2ut/blob/main/docs/PILOT.md)

## 1. Drive接続とpilot設定

抽出数・更新数・出力先を設定します。学習条件を変更する場合は新しいROOTを使用します。

In [ ]:
#@title 共通ログ：開始・逐次出力・生存通知・完了・失敗
from pathlib import Path
import json
"""Standard-library-only process logging, also embedded in Colab before install."""


def log_event(name, state, logs, detail=""):
    from datetime import datetime, timezone
    from pathlib import Path
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    line = f"{datetime.now(timezone.utc).isoformat()} [{name}] {state} {detail}\n"
    with (root / "notebook-events.log").open("a", encoding="utf-8") as stream:
        stream.write(line)
        stream.flush()
    print(line, end="", flush=True)


def run_process(command, logs, *, name="command", capture_json=False, env=None, cwd=None,
                heartbeat_seconds=10., extra_record=None):
    """Stream every child line; silence triggers liveness, never fake progress.

    stdout JSON is kept apart from stderr. Nested loggers share an owned POSIX
    process group so interruption cannot strand a second training process.
    """
    from datetime import datetime, timezone
    import hashlib
    import json
    import os
    from pathlib import Path
    import queue
    import signal
    import subprocess
    import tempfile
    import threading
    import time

    if heartbeat_seconds <= 0:
        raise ValueError("heartbeat_seconds must be positive")
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    command = list(map(str, command))
    key = hashlib.sha256(json.dumps(name, ensure_ascii=False).encode()).hexdigest()[:12] + "-" + str(time.time_ns())
    output, result_path, receipt = root / (key + ".log"), root / (key + ".stdout.log"), root / (key + ".json")
    launch_env = dict(os.environ if env is None else env)
    launch_env["PYTHONUNBUFFERED"] = "1"
    owns_group = os.name == "posix" and launch_env.get("AT_LOG_MANAGED_GROUP") != "1"
    if owns_group:
        launch_env["AT_LOG_MANAGED_GROUP"] = "1"
    started, clock = datetime.now(timezone.utc).isoformat(), time.perf_counter()
    record = dict(name=name, command=command, started_utc=started,
        host_utc_offset_seconds=int(datetime.now().astimezone().utcoffset().total_seconds()),
        elapsed_seconds=0., returncode=None, status="running", exception=None,
        pid=None, log=str(output.resolve()), stdout_log=str(result_path.resolve()) if capture_json else None,
        gpu_csv=None, gpu_monitor_error=None, cwd=str(Path(cwd or Path.cwd()).resolve()))
    lock, stop, done = threading.Lock(), threading.Event(), threading.Event()
    messages, reader_errors = queue.Queue(maxsize=256), []
    process = reader = result_stream = None
    stream = output.open("w", encoding="utf-8")

    def save_record():
        record["elapsed_seconds"] = time.perf_counter() - clock
        fd, temporary = tempfile.mkstemp(dir=root, prefix=".log-record-")
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as target:
                json.dump(record, target, ensure_ascii=False, indent=2, allow_nan=False)
                target.write("\n")
                target.flush()
            os.replace(temporary, receipt)
        finally:
            if os.path.exists(temporary):
                os.unlink(temporary)

    def notice(state, detail=""):
        line = f"[{name}] {state} pid={record['pid']} elapsed={time.perf_counter()-clock:.1f}s {detail}\n"
        with lock:
            stream.write(line)
            stream.flush()
        print(line, end="", flush=True)

    def read_lines(pipe):
        try:
            for line in pipe:
                with lock:
                    stream.write(line)
                    stream.flush()
                while not stop.is_set():
                    try:
                        messages.put(line, timeout=.1)
                        break
                    except queue.Full:
                        pass
        except Exception as error:
            reader_errors.append(error)
        finally:
            done.set()

    def terminate_child():
        if process is None:
            return
        if owns_group:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
        elif process.poll() is None:
            process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            if owns_group:
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            process.wait()
        if owns_group:
            # A nested child may outlive its exited leader after cancellation.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass

    try:
        notice("started", f"full log: {output.resolve()}")
        save_record()
        if capture_json:
            result_stream = result_path.open("w", encoding="utf-8")
        process = subprocess.Popen(command, stdout=result_stream if capture_json else subprocess.PIPE,
            stderr=subprocess.PIPE if capture_json else subprocess.STDOUT, text=True,
            encoding="utf-8", errors="replace", bufsize=1, env=launch_env, cwd=cwd,
            start_new_session=owns_group)
        record["pid"] = process.pid
        notice("running", "waiting for child output")
        save_record()
        pipe = process.stderr if capture_json else process.stdout
        reader = threading.Thread(target=read_lines, args=(pipe,), daemon=True, name="command-log-reader")
        reader.start()
        last_output = last_notice = time.perf_counter()
        while not (done.is_set() and messages.empty() and process.poll() is not None):
            try:
                line = messages.get(timeout=min(.1, heartbeat_seconds))
            except queue.Empty:
                line = None
            if line is not None:
                print(line, end="", flush=True)
                if line.strip():
                    last_output = time.perf_counter()
            if reader_errors:
                raise reader_errors[0]
            now = time.perf_counter()
            if now - max(last_output, last_notice) >= heartbeat_seconds:
                notice("running", f"no_output={now-last_output:.1f}s (liveness only; progress not confirmed)")
                save_record()
                last_notice = now
        code = process.wait()
        if reader_errors:
            raise reader_errors[0]
        if capture_json and code == 0:
            json.loads(result_path.read_text(encoding="utf-8"))
        record.update(returncode=code, status="succeeded" if code == 0 else "failed_or_interrupted")
        if code:
            raise subprocess.CalledProcessError(code, command)
        notice("completed", f"exit={code}; full log: {output.resolve()}")
    except BaseException as error:
        record.update(status="failed_or_interrupted", exception=type(error).__name__ + ": " + str(error))
        terminate_child()
        if process is not None:
            record["returncode"] = process.returncode
        notice("interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
               f"exit={record['returncode']} {type(error).__name__}; full log: {output.resolve()}")
        raise
    finally:
        stop.set()
        if reader is not None:
            reader.join(timeout=2)
        if process is not None and (reader is None or not reader.is_alive()):
            pipe = process.stderr if capture_json else process.stdout
            if pipe is not None:
                pipe.close()
        if result_stream is not None:
            result_stream.close()
        if extra_record is not None:
            record.update(extra_record() if callable(extra_record) else extra_record)
        save_record()
        stream.close()
    result = subprocess.CompletedProcess(command, record["returncode"],
        result_path.read_text(encoding="utf-8") if capture_json else None)
    result.record = record
    return result


def execute(*arguments, name=None, capture_json=False, env=None, cwd=None):
    command = list(map(str, arguments))
    if name is None:
        name = command[2].rsplit('.', 1)[-1] if len(command) > 2 and command[1] == '-m' else Path(command[0]).name
        if '--method' in command:
            name += ':' + command[command.index('--method') + 1]
        if '--prior-type' in command:
            name += ':' + command[command.index('--prior-type') + 1]
    return run_process(command, LOGS, name=name, capture_json=capture_json, env=env, cwd=cwd)


def execute_json(*arguments, name=None, env=None, cwd=None):
    return json.loads(execute(*arguments, name=name, capture_json=True, env=env, cwd=cwd).stdout)

LOGS = Path('/content/alignment-transfer-logs/pilot/bootstrap')


In [ ]:
from google.colab import drive
log_event('drive-mount', 'started', LOGS)
try:
    drive.mount('/content/drive')
except BaseException as error:
    log_event('drive-mount', 'failed', LOGS, type(error).__name__)
    raise
else:
    log_event('drive-mount', 'completed', LOGS)
import json, os, subprocess, sys, time
from pathlib import Path


In [ ]:
#@title 入力と出力先
REPO_URL = 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'
REPO_REVISION = '107605f4f5bd7b53722d73ced3fa5a76560e7a9d'
CORPUS = '/content/drive/MyDrive/ja-en-direct-s2st-corpus-data' #@param {type:"string"}
ACCEPTED = CORPUS + '/production/manifests/releases/accepted.jsonl'
ROOT = '/content/drive/MyDrive/alignment-transfer-s2ut-data/pilot-text-001' #@param {type:"string"}


In [ ]:
#@title 抽出数・学習・評価の設定
TRAIN_LIMIT = 10000 #@param {type:"integer"}
DEV_LIMIT = 200 #@param {type:"integer"}
SEED = 1 #@param {type:"integer"}
BATCH_SIZE = 8 #@param {type:"integer"}
UPDATE_FREQUENCY = 4 #@param {type:"integer"}
MAX_TOKENS = 20000 #@param {type:"integer"}
LOADER_WORKERS = 2 #@param {type:"integer"}
PREFETCH_BATCHES = 2 #@param {type:"integer"}
LOCAL_CACHE = '/content/alignment-transfer-cache' #@param {type:"string"}
CACHE_GB = 8.0 #@param {type:"number"}
T2TT_UPDATES = 20000 #@param {type:"integer"}
S2UT_UPDATES = 10000 #@param {type:"integer"}
ANNEAL_STEPS = 5000 #@param {type:"integer"}
RUN_CUDA_INTEGRATION = True #@param {type:"boolean"}
GENERATE_WAVEFORMS = False #@param {type:"boolean"}
ASR_BLEU = False #@param {type:"boolean"}


In [ ]:
#@title 内部パスと設定検査
PYTHON = '/content/alignment-venv/bin/python'
REPO = Path('/content/alignment-transfer-s2ut')
DATA = Path(ROOT) / 'data'
MANIFEST = DATA / 'manifest.jsonl'
RUNS = Path(ROOT) / 'runs'
LOGS = Path('/content/alignment-transfer-logs/pilot') / Path(ROOT).name
CONDITIONS = ['baseline', 'annealed_monotonic', 'annealed_t2tt']
assert not Path(ROOT).resolve().is_relative_to(Path(CORPUS).resolve())
assert Path(ACCEPTED).is_file(), ACCEPTED
assert 1 <= ANNEAL_STEPS < S2UT_UPDATES
assert not ASR_BLEU or GENERATE_WAVEFORMS, 'ASR-BLEU requires generated waveforms'
os.environ['HF_HOME'] = '/content/huggingface'
os.environ['MPLBACKEND'] = 'Agg'
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['MPLCONFIGDIR'] = '/content/matplotlib-cache'


## 2. 環境の構築と検査

Python 3.10.22 / torch・torchaudio 2.7.1 / 固定fairseqを復元します。
既存ROOTに別のコード・seed・データ条件を混ぜず、変更時は新しいROOTを使ってください。
すべてのコマンドは開始・PID・途中出力・完了／失敗を表示し、`LOGS`へ保存します。無出力が10秒続いた場合は生存通知を表示します。生存通知だけで実際の進捗が確認できたとは扱いません。


In [ ]:
#@title Python・fairseq環境を構築
if not (REPO / '.git').is_dir():
    execute('git', 'clone', '--no-checkout', REPO_URL, str(REPO), name='clone-alignment')
execute('git', '-C', str(REPO), 'fetch', 'origin', REPO_REVISION, name='fetch-alignment')
execute('git', '-C', str(REPO), 'checkout', '--detach', REPO_REVISION, name='checkout-alignment')
os.chdir(REPO)
execute(sys.executable, '-m', 'pip', 'install', 'uv', name='install-uv')
execute('uv', 'venv', '--python', '3.10.22', '--seed', '--allow-existing', '/content/alignment-venv', name='create-python-environment')
for packages in [
    ['pip==23.3.2', 'setuptools<81', 'wheel'],
    ['torch==2.7.1', 'torchaudio==2.7.1', 'numpy==1.26.4', 'Cython==3.2.9'],
    ['--no-build-isolation', '-r', 'requirements-fairseq.txt'],
    ['-r', 'requirements.txt'],
]:
    execute(PYTHON, '-m', 'pip', 'install', *packages)
execute(PYTHON, '-m', 'pip', 'check', name='dependency-check')


In [ ]:
def run(name, module, *arguments, gpu=False, env=None):
    command = [PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'run', '--name', name, '--logs', str(LOGS)]
    if gpu:
        command.append('--monitor-gpu')
    command += ['--', PYTHON, '-m', module, *map(str, arguments)]
    return execute(*command, name='notebook:' + name, env=env)


In [ ]:
run('runtime', 'alignment_transfer_s2ut.pilot', 'runtime')
runtime = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'runtime', name='runtime-details')
(Path(ROOT) / 'runtime.json').write_text(json.dumps(runtime, indent=2), encoding='utf-8')
print(runtime)
run('cpu-tests', 'pytest', '-v', '--tb=short', '-m', 'not gpu')


## 3. 固定したpilot manifest

train最大10,000件・dev最大200件を元のsplit内でseed固定抽出します。件数は上で変更できます。
音声hashを計算するのは選択したpairだけです。選択結果・元manifestのhash・population件数を保存し、条件変更を伴うresumeを拒否します。

In [ ]:
run('prepare-data', 'alignment_transfer_s2ut.pilot', 'prepare', '--input', ACCEPTED,
    '--corpus-root', CORPUS, '--output', DATA, '--train-limit', TRAIN_LIMIT, '--dev-limit', DEV_LIMIT,
    '--seed', SEED, '--resume')
selection = json.loads((DATA / 'selection.json').read_text())
print({key: selection[key] for key in ['population_counts', 'selected_counts', 'test_used', 'manifest_hash']})
rows = [json.loads(line) for line in MANIFEST.read_text().splitlines() if line.strip()]
run('architecture', 'alignment_transfer_s2ut.compare_architectures', '--t2tt-config', 'configs/t2tt/fisher.yaml',
    '--s2ut-config', 'configs/s2ut/fisher.yaml', '--output', Path(ROOT) / 'architecture.json', '--resume')


## 4. 実モデルのCUDA接続試験

選択済みデータのtrain最大2件・dev1件で実CTC / HuBERT / strongの3条件 / 公式vocoderを確認します。
2更新の接続試験で、翻訳品質の評価ではありません。テスト用出力は試行ごとに別フォルダへ保存します。

In [ ]:
if RUN_CUDA_INTEGRATION:
    tiny = [row for row in rows if row['split'] == 'train'][:2] + [row for row in rows if row['split'] == 'dev'][:1]
    integration = Path(ROOT) / 'integration'
    integration.mkdir(parents=True, exist_ok=True)
    tiny_manifest = integration / 'input.jsonl'
    tiny_manifest.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in tiny), encoding='utf-8')
    environment = os.environ.copy()
    environment['AT_S2UT_GPU_MANIFEST'] = str(tiny_manifest)
    run('cuda-integration', 'pytest', 'tests/test_external_pipeline.py', '-m', 'gpu', '-q', '-s',
        '--basetemp', integration / ('attempt-' + str(time.time_ns())), gpu=True, env=environment)
else:
    log_event('cuda-integration', 'skipped', LOGS, 'RUN_CUDA_INTEGRATION=False')


## 5. T2TTとalignment / unitの前処理

T2TTは同じpilot train pairから別の更新数で学習します。2更新の接続試験用teacherを本pilotへ流用しません。
最良dev lossのT2TT checkpointから、参照訳なしの生成でdev全件のtext BLEUも測定します。BLEU・EOS到達率・原文/予測/参照を確認してから前処理へ進んでください。学習途中のteacherによるpilotから、転移効果の最終結論は出しません。

In [ ]:
run('t2tt', 'alignment_transfer_s2ut.train_t2tt', '--manifest', MANIFEST, '--config', 'configs/t2tt/fisher.yaml',
    '--output', Path(ROOT) / 't2tt', '--max-updates', T2TT_UPDATES, '--warmup-updates', 1000,
    '--seed', SEED, '--batch-size', BATCH_SIZE, '--update-frequency', UPDATE_FREQUENCY, '--max-tokens', MAX_TOKENS,
    '--loader-workers', LOADER_WORKERS, '--prefetch-batches', PREFETCH_BATCHES, '--pin-memory',
    '--validation-interval', 500, '--save-interval-updates', 500, '--device', 'cuda', '--resume-if-exists', gpu=True)
run('t2tt-dev-text', 'alignment_transfer_s2ut.evaluate_t2tt', '--manifest', MANIFEST, '--split', 'dev',
    '--checkpoint', Path(ROOT) / 't2tt/checkpoint_best.pt', '--beam-size', 5, '--max-chars', 1000,
    '--length-penalty', 1.0, '--output', Path(ROOT) / 't2tt/dev-text-metrics.json', '--device', 'cuda', '--resume', gpu=True)
text_metrics = json.loads((Path(ROOT) / 't2tt/dev-text-metrics.json').read_text())
print({key: text_metrics[key] for key in ['text_bleu', 'bleu_signature', 'samples', 'eos_reached_ratio', 'brevity_penalty']})
text_predictions = [json.loads(line) for line in (Path(ROOT) / 't2tt/dev-text-metrics-predictions.jsonl').read_text().splitlines() if line.strip()]
dev_text_by_id = {row['pair_id']: row['en_text'] for row in rows if row['split'] == 'dev'}
for prediction in text_predictions[:5]:
    print('JA:', prediction['source_text'], '\nHYP:', prediction['hypothesis'], '\nREF:', dev_text_by_id[prediction['pair_id']])
run('mt-attention', 'alignment_transfer_s2ut.extract_t2tt_attention', '--manifest', MANIFEST,
    '--checkpoint', Path(ROOT) / 't2tt/checkpoint_best.pt', '--output', Path(ROOT) / 'mt', '--device', 'cuda', '--resume', gpu=True)
run('ctc', 'alignment_transfer_s2ut.align_speech_text', '--manifest', MANIFEST, '--config', 'configs/alignment/ctc.yaml',
    '--output', Path(ROOT) / 'ctc', '--device', 'cuda', '--resume', gpu=True)
run('km100', 'alignment_transfer_s2ut.fetch_assets', '--config', 'configs/units/hubert_km100.yaml')
run('units', 'alignment_transfer_s2ut.extract_units', '--manifest', MANIFEST, '--config', 'configs/units/hubert_km100.yaml',
    '--output', Path(ROOT) / 'units', '--device', 'cuda', '--resume', gpu=True)
run('teacher', 'alignment_transfer_s2ut.build_teacher_alignment', '--manifest', MANIFEST,
    '--mt-attention', Path(ROOT) / 'mt', '--alignment', Path(ROOT) / 'ctc', '--units', Path(ROOT) / 'units',
    '--config', 'configs/s2ut/fisher.yaml', '--output', Path(ROOT) / 'teacher', '--resume')


## 6. Alignment診断：trainとdev

各split最大500件・20 heatmapを保存します。coverage、invalid row、raw / smoothed entropyを確認し、
音声と元の文字列を並べてCTCの対応を確認します。coverageの高さだけで時刻精度が正しいとは判断しません。

In [ ]:
from IPython.display import display, Image, Audio
by_id = {row['pair_id']: row for row in rows}
for split in ['train', 'dev']:
    audit = Path(ROOT) / ('alignment-audit-' + split)
    run('audit-' + split, 'alignment_transfer_s2ut.diagnose_alignment', '--manifest', MANIFEST,
        '--teacher', Path(ROOT) / 'teacher', '--units', Path(ROOT) / 'units', '--config', 'configs/s2ut/fisher.yaml',
        '--split', split, '--max-samples', 500, '--heatmaps', 20, '--seed', SEED, '--source-mapping', 'frontend',
        '--prior-smoothing', 0.05, '--output', audit, '--overwrite')
    summary = json.loads((audit / 'summary.json').read_text())
    print(split, {key: summary[key] for key in ['audited_pairs', 'valid_source_tokens_ratio',
        'valid_target_tokens_ratio', 'valid_unit_rows_ratio', 'prior_entropy', 'smoothed_entropy', 'zero_cell_ratio']})
    assert summary['counts']['valid_unit_rows'] > 0, 'No usable alignment rows; inspect CTC and teacher first'
    pairs = [json.loads(line) for line in (audit / 'pairs.jsonl').read_text().splitlines() if line.strip()]
    # Display two examples per split; all 20 PNG/NPZ pairs remain in the audit directory.
    import hashlib
    for pair in pairs[:2]:
        row = by_id[pair['pair_id']]
        filename = hashlib.sha256(json.dumps(row['pair_id'], ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
        print(row['pair_id'], row['ja_text'], '/', row['en_text'])
        display(Image(filename=str(audit / 'heatmaps' / (filename + '.png'))))
        display(Audio(filename=row['ja_audio']), Audio(filename=row['en_audio']))


## 7. Strongの3条件pilot

全条件で同じ初期化・data・batch・学習率・warmup 1,000・更新数を使います。
既定10,000更新のうち、annealingは5,000更新で終了し、後半はbeta=0です。
planセルは実行予定を保存します。trainセルはcheckpointから自動再開し、失敗した条件で停止します。

少量の入力が揃うと学習を開始し、CPU workerが次の入力を先読みします。未拡張FBANK/CMVNとunit/teacherはローカルSSDの上限付きcacheへ保存し、3条件で共有します。キャッシュは全件コピーせず、必要時に作ります。SpecAugmentは学習threadでGPU転送前に適用します。

In [ ]:
PILOT_ARGS = ['--manifest', str(MANIFEST), '--config', 'configs/s2ut/fisher.yaml', '--units', str(Path(ROOT) / 'units'),
    '--teacher', str(Path(ROOT) / 'teacher'), '--output', str(RUNS), '--logs', str(LOGS),
    '--max-updates', str(S2UT_UPDATES), '--anneal-steps', str(ANNEAL_STEPS), '--seed', str(SEED),
    '--batch-size', str(BATCH_SIZE), '--update-frequency', str(UPDATE_FREQUENCY), '--max-tokens', str(MAX_TOKENS), '--loader-workers', str(LOADER_WORKERS),
    '--prefetch-batches', str(PREFETCH_BATCHES), '--cache-dir', LOCAL_CACHE, '--cache-gb', str(CACHE_GB)]
plan = execute(PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'plan', *PILOT_ARGS, name='pilot-plan', capture_json=True).stdout
(Path(ROOT) / 'pilot-plan.json').write_text(plan, encoding='utf-8')
print(plan)


### 学習を実行

直前のplanを確認した後に実行します。3条件は順番に学習し、各条件のcheckpointから自動再開します。

In [ ]:
execute(PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'train', *PILOT_ARGS)
run('comparison', 'alignment_transfer_s2ut.compare_experiments', *[RUNS / name for name in CONDITIONS],
    '--output', Path(ROOT) / 'fair-comparison.json')


## 8. 共通dev評価と集計

dev全件で同じbeam 5 / max units 1,000 / beta=0を使用します。
既定はunit error rate。冒頭でGENERATE_WAVEFORMSとASR_BLEUを有効にすると、共通の固定vocoder・ASRで音声とASR-BLEUも評価します。
pilot後の調整にdevを使い、最終test評価は別途行います。

In [ ]:
speech_only = Path(ROOT) / 'dev-speech-only.jsonl'
run('dev-input', 'alignment_transfer_s2ut.prepare_inference_input', '--manifest', MANIFEST,
    '--split', 'dev', '--output', speech_only, '--resume')
if GENERATE_WAVEFORMS:
    run('vocoder', 'alignment_transfer_s2ut.fetch_assets', '--config', 'configs/s2ut/vocoder.yaml')
for name in CONDITIONS:
    folder = RUNS / name
    vocoder = ['--vocoder-config', 'configs/s2ut/vocoder.yaml'] if GENERATE_WAVEFORMS else ['--units-only']
    run(name + '-dev-infer', 'alignment_transfer_s2ut.infer', '--input', speech_only,
        '--checkpoint', folder / 'checkpoint_best.pt', '--beam-size', 5, '--max-units', 1000,
        '--device', 'cuda', '--output', folder / 'dev-predictions.jsonl', '--resume', *vocoder, gpu=True)
    asr = ['--asr-config', 'configs/experiments/evaluation.yaml', '--device', 'cuda'] if ASR_BLEU else []
    run(name + '-dev-evaluate', 'alignment_transfer_s2ut.evaluate', '--manifest', MANIFEST, '--split', 'dev',
        '--predictions', folder / 'dev-predictions.jsonl', '--units', Path(ROOT) / 'units',
        '--output', folder / 'dev-metrics.json', '--resume', *asr, gpu=ASR_BLEU)
run('pilot-report', 'alignment_transfer_s2ut.pilot', 'report', '--runs', RUNS, '--logs', LOGS,
    '--t2tt-evaluation', Path(ROOT) / 't2tt/dev-text-metrics.json', '--output', Path(ROOT) / 'pilot-summary.json')
summary = json.loads((Path(ROOT) / 'pilot-summary.json').read_text())
display(summary)
display(Image(filename=str(Path(ROOT) / 'pilot-summary.png')))

if not GENERATE_WAVEFORMS:
    log_event('waveform-generation', 'skipped', LOGS, 'GENERATE_WAVEFORMS=False')
if not ASR_BLEU:
    log_event('asr-bleu', 'skipped', LOGS, 'ASR_BLEU=False')


## 9. 保存される結果

- `data/selection.json`：元のmanifest hash・抽出条件・元splitと使用件数
- `t2tt/dev-text-metrics.json`, `t2tt/dev-text-metrics-predictions.jsonl`：dev text BLEU・SacreBLEU署名・EOS到達率・生成設定・予測文
- `alignment-audit-train/`, `alignment-audit-dev/`：coverage・entropy・sample別指標・20 heatmapずつ
- `runs/{condition}/`：checkpoint・学習metrics・dev予測と評価
- `LOGS`（VMローカル）：各試行のコマンド・終了状態・所要秒数・完全ログ・5秒間隔のGPU使用率/メモリCSV
- `runtime.json`：Python・torch・CUDA・fairseq revisionとGPU機種/メモリ
- `fair-comparison.json`, `pilot-summary.json`, `pilot-summary.png`：比較条件検査・最良dev loss・beta・更新数・dev評価・記録時間・dev loss曲線

失敗時はLOGSのログを確認し、同じ設定で該当セルから再開できます。
GPU使用率が低い時間帯を学習ログと照合し、FBANK/I/Oの最適化が必要か判断します。
このノートブックは実行準備です。実コーパス上の効果は、実行して得られた結果で判断してください。

`input/data_wait_seconds`は入力を待った時間、`input/optimization_seconds`はCPU augmentation・GPU転送・forward/backward・optimizerの時間です。validationとcheckpoint保存は含みません。LOCAL_CACHEはVM上の一時領域で、runtime再作成後は必要な分から再取得します。